# Блок Прогноз — Regularized Logistic Regression + трансформации

Итоговая карта строится ML-моделью `LogisticRegression.predict_proba()` по нелинейно преобразованным геологическим признакам.

In [ ]:
# %% Cell 0
import json
import os
import re
import shutil
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm
from matplotlib.patches import Patch
from pyproj import CRS
from shapely.geometry import box
from shapely.ops import unary_union
from shapely.prepared import prep
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")

In [ ]:
# %% Cell 1


# =========================
# SETTINGS
# =========================
CELL_SIZE = 500
RANDOM_STATE = 42
USE_SUPERVISED = True

# Regularized Logistic Regression — основная ML-модель прогноза.
# Итоговая поверхность строится через predict_proba(...),
# а геологическая логика задается не финальной суммой, а признаками: exp(-distance/scale),
# степенными преобразованиями и взаимодействиями факторов.
LOGREG_C = 0.55
LOGREG_MAX_ITER = 8000

# Радиус влияния известных проявлений/геохимических точек для формирования обучающих меток.
# Отсутствие точки не означает отсутствие руды, поэтому отрицательные объекты берутся только
# как надежно удаленные от известных проявлений.
EVIDENCE_RADIUS_CELLS = 6.0
POSITIVE_EVIDENCE_Q = 0.86
NEGATIVE_EVIDENCE_Q = 0.22

# Сглаживание ML-результата, чтобы убрать пиксельный шум.
ML_SMOOTH_PASSES = 2

# Нелинейное усиление максимумов вероятности. 1.0 — без усиления.
PROSPECTIVITY_GAMMA = 1.25

Q_FACIES = 0.78
Q_PALEO = 0.76
Q_STRUCT = 0.72
Q_MAGM = 0.42
Q_TECT1 = 0.74
Q_TECT2 = 0.74

N_DISPLAY_CLASSES = 20
SHOW_POINTS = False

In [ ]:
# %% Cell 2

# =========================
# PATHS
# =========================
def find_base_dir() -> Path:
    candidates = [
        Path.cwd(),
        Path("/mnt/data/prog_zip"),
        Path("/mnt/data"),
        Path(r"C:\Users\janfi\OneDrive\Desktop\Прочее\Прогноз"),
    ]
    for base in candidates:
        shp_dir = base / "shp_dbf"
        if shp_dir.exists() and (shp_dir / "svita_new.shp").exists():
            return base
    raise FileNotFoundError("Не найден каталог с shp_dbf.")

BASE_DIR = find_base_dir()
SHP_DIR = BASE_DIR / "shp_dbf"
OUT_DIR = BASE_DIR / "logreg_transform_result"
OUT_DIR.mkdir(parents=True, exist_ok=True)
SAFE_ALIAS_DIR = OUT_DIR / "_safe_aliases"
SAFE_ALIAS_DIR.mkdir(parents=True, exist_ok=True)

OUT_GPKG = OUT_DIR / "forecast_logreg_transform.gpkg"
OUT_PNG = OUT_DIR / "forecast_logreg_transform.png"
OUT_COMPARE = OUT_DIR / "compare_logreg_transform.png"
OUT_PROX = OUT_DIR / "prox_magm_logreg_transform.png"
OUT_CSV = OUT_DIR / "grid_logreg_transform.csv"
OUT_JSON = OUT_DIR / "metrics_logreg_transform.json"

In [ ]:
# %% Cell 3

# =========================
# HELPERS
# =========================
def normalize_01(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    mn = np.nanmin(arr[finite])
    mx = np.nanmax(arr[finite])
    if np.isclose(mx, mn):
        return np.full_like(arr, 0.5, dtype=float)
    out = np.full_like(arr, np.nan, dtype=float)
    out[finite] = (arr[finite] - mn) / (mx - mn)
    return out

def robust_normalize_01(values, q_low=0.03, q_high=0.97):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return np.zeros_like(arr, dtype=float)
    lo = np.nanquantile(arr[finite], q_low)
    hi = np.nanquantile(arr[finite], q_high)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        return normalize_01(arr)
    return np.clip((arr - lo) / (hi - lo), 0, 1)

def smooth_on_regular_grid(grid, value_col, shape, passes=1):
    try:
        from scipy.signal import convolve2d
    except Exception:
        return grid[value_col].to_numpy()
    arr = np.full(shape, np.nan, dtype=float)
    arr[grid["row"].to_numpy(), grid["col"].to_numpy()] = grid[value_col].to_numpy()
    kernel = np.array([[1.0, 1.2, 1.0], [1.2, 3.0, 1.2], [1.0, 1.2, 1.0]], dtype=float)
    smoothed = arr.copy()
    for _ in range(max(1, passes)):
        valid = np.isfinite(smoothed).astype(float)
        filled = np.nan_to_num(smoothed, nan=0.0)
        num = convolve2d(filled, kernel, mode="same", boundary="fill", fillvalue=0)
        den = convolve2d(valid, kernel, mode="same", boundary="fill", fillvalue=0)
        smoothed = np.where(den > 0, num / den, np.nan)
    return smoothed[grid["row"].to_numpy(), grid["col"].to_numpy()]

def local_max_mask(grid, value_col, shape):
    try:
        from scipy.ndimage import maximum_filter
    except Exception:
        vals = grid[value_col].to_numpy()
        thr = np.nanquantile(vals, 0.98)
        return vals >= thr
    arr = np.full(shape, np.nan, dtype=float)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    vals = grid[value_col].to_numpy()
    arr[rows, cols] = vals
    filled = np.nan_to_num(arr, nan=-9999.0)
    locmax = maximum_filter(filled, size=3, mode="nearest")
    return (np.isfinite(arr) & (filled >= locmax))[rows, cols]

def keep_large_components(grid, bool_col, shape, min_cells=4):
    try:
        from scipy import ndimage
    except Exception:
        return grid[bool_col].to_numpy().astype(bool)
    arr = np.zeros(shape, dtype=np.uint8)
    rows = grid["row"].to_numpy()
    cols = grid["col"].to_numpy()
    arr[rows, cols] = grid[bool_col].to_numpy().astype(np.uint8)
    structure = np.ones((3, 3), dtype=np.uint8)
    labeled, _ = ndimage.label(arr, structure=structure)
    sizes = np.bincount(labeled.ravel())
    keep = np.isin(labeled, np.where(sizes >= min_cells)[0]) & (labeled > 0)
    return keep[rows, cols]

def read_sidecar_proj4(shp_path: Path):
    sidecar = shp_path.with_name(shp_path.stem + "_shp.pj4")
    if sidecar.exists():
        txt = sidecar.read_text(encoding="utf-8", errors="ignore")
        m = re.search(r"pj4=(.+)", txt)
        if m:
            return m.group(1).strip()
    return None

def prepare_ascii_aliases(shp_dir: Path, alias_dir: Path):
    aliases, stems = {}, {}
    for name_b in os.listdir(os.fsencode(shp_dir)):
        if not name_b.endswith((b".shp", b".shx", b".dbf", b".prj", b".pj4")) or name_b.endswith(b"_shp.pj4"):
            continue
        base_b, ext_b = os.path.splitext(name_b)
        stems.setdefault(base_b, set()).add(ext_b)

    alias_idx = 0
    for base_b, exts in sorted(stems.items()):
        try:
            base_s = os.fsdecode(base_b)
            safe = all(ord(ch) < 128 and (ch.isalnum() or ch in "_-. ") for ch in base_s)
        except Exception:
            safe = False
            base_s = None

        if safe:
            aliases[base_s] = shp_dir / f"{base_s}.shp"
            continue

        alias = f"layer_{alias_idx:02d}"
        alias_idx += 1
        for ext_b in exts:
            src = os.path.join(os.fsencode(shp_dir), base_b + ext_b)
            dst = alias_dir / f"{alias}{os.fsdecode(ext_b)}"
            shutil.copyfile(src, dst)
        pj4_src = os.path.join(os.fsencode(shp_dir), base_b + b"_shp.pj4")
        if os.path.exists(pj4_src):
            shutil.copyfile(pj4_src, alias_dir / f"{alias}_shp.pj4")
        aliases[alias] = alias_dir / f"{alias}.shp"
    return aliases

def load_layer(path: Path):
    gdf = gpd.read_file(path)
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf = gdf[~gdf.geometry.is_empty].copy()
    if gdf.crs is None:
        proj4 = read_sidecar_proj4(path)
        if proj4:
            gdf = gdf.set_crs(CRS.from_proj4(proj4), allow_override=True)
    return gdf

def to_crs_safe(gdf, target_crs):
    if gdf.crs is None and target_crs is not None:
        return gdf.set_crs(target_crs, allow_override=True)
    if target_crs is None or gdf.crs == target_crs:
        return gdf
    return gdf.to_crs(target_crs)

def build_grid(mask, cell_size):
    mask_union = unary_union(mask.geometry)
    prepared_mask = prep(mask_union)
    minx, miny, maxx, maxy = mask.total_bounds
    xs = np.arange(minx, maxx, cell_size)
    ys = np.arange(miny, maxy, cell_size)
    rows = []
    cell_id = 0
    for r, y in enumerate(ys):
        for c, x in enumerate(xs):
            geom = box(x, y, x + cell_size, y + cell_size)
            if prepared_mask.intersects(geom):
                rows.append((cell_id, r, c, geom))
                cell_id += 1
    grid = gpd.GeoDataFrame(rows, columns=["cell_id", "row", "col", "geometry"], geometry="geometry", crs=mask.crs)
    return grid, mask_union, (len(ys), len(xs))

def add_distance_feature(grid, source, name):
    source_union = unary_union(source.geometry)
    d = np.empty(len(grid), dtype=float)
    for i, geom in enumerate(grid.geometry.values):
        d[i] = 0.0 if geom.intersects(source_union) else geom.distance(source_union)
    grid[name] = d
    return grid

def distance_to_proximity(distance, transform="sqrt", q=0.75):
    d = np.asarray(distance, dtype=float)
    d = np.clip(d, 0, None)
    if transform == "sqrt":
        t = np.sqrt(d)
    elif transform == "cbrt":
        t = np.cbrt(d)
    else:
        t = d
    scale = float(np.nanquantile(t, q))
    if not np.isfinite(scale) or scale <= 0:
        scale = max(float(np.nanmean(t)), 1.0)
    return np.clip(np.exp(-t / scale), 0, 1)

def collect_points(mask_crs, aliases):
    base_names = {"svita_new", "fasii", "glub_raz_nw", "glub_r_nw", "gr_dol_vp_poly", "kory", "dayki_buf"}
    layers = []
    for name, shp_path in aliases.items():
        if name in base_names:
            continue
        gdf = to_crs_safe(load_layer(shp_path), mask_crs)
        types = {str(x) for x in gdf.geom_type.unique()}
        if "Point" in types or "MultiPoint" in types:
            layers.append(gdf)
    if not layers:
        return None
    pts = pd.concat(layers, ignore_index=True)
    return gpd.GeoDataFrame(pts, geometry="geometry", crs=mask_crs)

def make_display_classes(grid):
    disp = robust_normalize_01(grid["prospectivity"].to_numpy(), 0.02, 0.98)
    grid["display_score"] = disp
    bins = np.linspace(0, 1, N_DISPLAY_CLASSES + 1)
    grid["display_class"] = np.digitize(disp, bins[1:-1], right=False)
    return grid

def set_mask_extent(ax, mask):
    minx, miny, maxx, maxy = mask.total_bounds
    padx = (maxx - minx) * 0.02
    pady = (maxy - miny) * 0.02
    ax.set_xlim(minx - padx, maxx + padx)
    ax.set_ylim(miny - pady, maxy + pady)

def mark_gold_zones(grid, shape, mask_union):
    """
    Выделение наиболее перспективных зон для отображения.
    Важно: это НЕ отдельный фактор итогового прогноза.
    Итоговый прогноз уже рассчитан как MLP-предсказание в поле prospectivity.
    Здесь только выбираются верхние локальные области для жёлтой подсветки.
    """
    q_core = float(grid["prospectivity"].quantile(0.955))
    q_support = float(grid["prospectivity"].quantile(0.90))

    support = grid["prospectivity"] >= q_support
    local_peak = local_max_mask(grid, "prospectivity", shape)

    # Небольшая геологическая проверка, чтобы жёлтые зоны не появлялись
    # совсем без поддержки тектоники/магматизма/структурного фактора.
    support_geo = (
        (grid["tect_magm_intersection"] >= grid["tect_magm_intersection"].quantile(0.72)) |
        (grid["tect_struct_intersection"] >= grid["tect_struct_intersection"].quantile(0.72)) |
        (grid["coincidence_score"] >= grid["coincidence_score"].quantile(0.72))
    )

    grid["gold_zone_raw"] = (
        (grid["prospectivity"] >= q_core) |
        (support & local_peak & support_geo)
    ).astype(int)

    grid["gold_zone"] = keep_large_components(grid, "gold_zone_raw", shape, min_cells=4).astype(int)
    return grid


def plot_prox(grid, mask, out_png):
    fig, ax = plt.subplots(figsize=(8, 8))
    grid.plot(column="prox_magm", ax=ax, cmap="RdYlBu_r", linewidth=0, legend=True)
    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)
    set_mask_extent(ax, mask)
    ax.set_title("Магматогенный фактор: proximity")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_final(grid, mask, points, out_png):
    fig, ax = plt.subplots(figsize=(10, 10))
    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr_r.N)
    grid.plot(column="display_class", ax=ax, cmap="bwr_r", norm=norm, linewidth=0, legend=True)

    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=ax, color="#f2d200", linewidth=0)

    mask.boundary.plot(ax=ax, color="black", linewidth=0.5)

    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=ax, color="yellow", markersize=8, edgecolor="black", linewidth=0.25)

    ax.legend(handles=[Patch(facecolor="#f2d200", edgecolor="black", label="Gold zones")], loc="lower right", frameon=True)
    set_mask_extent(ax, mask)
    ax.set_title("Итоговый прогноз")
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

def plot_compare(grid, mask, points, out_png):
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    grid.plot(column="prox_magm", ax=axes[0], cmap="RdYlBu_r", linewidth=0)
    mask.boundary.plot(ax=axes[0], color="black", linewidth=0.5)
    set_mask_extent(axes[0], mask)
    axes[0].set_title("prox_magm")
    axes[0].set_axis_off()

    bins = np.arange(N_DISPLAY_CLASSES + 1)
    norm = BoundaryNorm(bins, plt.cm.bwr_r.N)
    grid.plot(column="display_class", ax=axes[1], cmap="bwr_r", norm=norm, linewidth=0, legend=True)

    gold = grid[grid["gold_zone"] == 1]
    if len(gold) > 0:
        gold.plot(ax=axes[1], color="#f2d200", linewidth=0)

    mask.boundary.plot(ax=axes[1], color="black", linewidth=0.5)

    if SHOW_POINTS and points is not None and len(points) > 0:
        points.plot(ax=axes[1], color="yellow", markersize=8, edgecolor="black", linewidth=0.25)

    axes[1].legend(handles=[Patch(facecolor="#f2d200", edgecolor="black", label="Gold zones")], loc="lower right", frameon=True)
    set_mask_extent(axes[1], mask)
    axes[1].set_title("Итоговый прогноз")
    axes[1].set_axis_off()

    plt.tight_layout()
    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.close()

In [ ]:
# %% Cell 4
# =========================
# LOAD DATA
# =========================
aliases = prepare_ascii_aliases(SHP_DIR, SAFE_ALIAS_DIR)

mask = load_layer(aliases["svita_new"])
facies = to_crs_safe(load_layer(aliases["fasii"]), mask.crs)
tect1 = to_crs_safe(load_layer(aliases["glub_raz_nw"]), mask.crs)
tect2 = to_crs_safe(load_layer(aliases["glub_r_nw"]), mask.crs)
paleo = to_crs_safe(load_layer(aliases["gr_dol_vp_poly"]), mask.crs)
struct = to_crs_safe(load_layer(aliases["kory"]), mask.crs)
magm = to_crs_safe(load_layer(aliases["dayki_buf"]), mask.crs)
points = collect_points(mask.crs, aliases)

In [ ]:
# %% Cell 5
# =========================
# GRID + FEATURES
# =========================
grid, mask_union, grid_shape = build_grid(mask, CELL_SIZE)

for src, name in [
    (facies, "dist_facies"),
    (paleo, "dist_paleo"),
    (struct, "dist_struct"),
    (magm, "dist_magm"),
    (tect1, "dist_tect1"),
    (tect2, "dist_tect2"),
]:
    grid = add_distance_feature(grid, src, name)

grid["prox_facies"] = distance_to_proximity(grid["dist_facies"], "cbrt", Q_FACIES)
grid["prox_paleo"] = distance_to_proximity(grid["dist_paleo"], "cbrt", Q_PALEO)
grid["prox_struct"] = distance_to_proximity(grid["dist_struct"], "sqrt", Q_STRUCT)
grid["prox_magm"] = distance_to_proximity(grid["dist_magm"], "sqrt", Q_MAGM)
grid["prox_tect1"] = distance_to_proximity(grid["dist_tect1"], "cbrt", Q_TECT1)
grid["prox_tect2"] = distance_to_proximity(grid["dist_tect2"], "cbrt", Q_TECT2)

grid["tect_combo"] = 0.5 * (grid["prox_tect1"] + grid["prox_tect2"])
grid["tect_intersection"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["tect_magm_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_magm"])
grid["tect_struct_intersection"] = np.sqrt(grid["tect_combo"] * grid["prox_struct"])
grid["paleo_struct_intersection"] = np.sqrt(grid["prox_paleo"] * grid["prox_struct"])

combo_core = (
    np.clip(grid["tect_combo"], 0, 1)
    * np.clip(0.55 * grid["prox_magm"] + 0.45 * grid["prox_struct"], 0, 1)
    * np.clip(0.60 * grid["prox_paleo"] + 0.40 * grid["prox_facies"], 0, 1)
)
grid["coincidence_score"] = robust_normalize_01(np.sqrt(np.clip(combo_core, 0, 1)), 0.02, 0.98)

tect_support = 0.40 * grid["prox_magm"] + 0.35 * grid["prox_struct"] + 0.25 * grid["prox_paleo"]
grid["tect_only_penalty"] = robust_normalize_01(np.clip(grid["tect_combo"] - tect_support, 0, 1), 0.02, 0.98)

In [ ]:
# %% Cell 6
# =========================
# GEO SCORE
# =========================
grid["geo_score_raw"] = (
    0.14 * grid["prox_tect1"] +
    0.14 * grid["prox_tect2"] +
    0.14 * grid["prox_paleo"] +
    0.11 * grid["prox_struct"] +
    0.08 * grid["prox_facies"] +
    0.08 * grid["prox_magm"] +
    0.08 * grid["tect_intersection"] +
    0.08 * grid["tect_magm_intersection"] +
    0.05 * grid["tect_struct_intersection"] +
    0.04 * grid["paleo_struct_intersection"] +
    0.10 * grid["coincidence_score"] -
    0.08 * grid["tect_only_penalty"]
)
grid["geo_score"] = robust_normalize_01(grid["geo_score_raw"], 0.02, 0.98)
grid["geo_score_sm"] = robust_normalize_01(smooth_on_regular_grid(grid, "geo_score", grid_shape, passes=2), 0.02, 0.98)

In [ ]:
# %% Cell 7
# =========================
# REGULARIZED LOGISTIC REGRESSION + TRANSFORMATIONS — ML ОСНОВА ПРОГНОЗА
# =========================

# Идея метода:
# 1) расстояния до геологических факторов переводятся в близость/proximity;
# 2) добавляются нелинейные признаки и взаимодействия факторов;
# 3) L2-регуляризованная логистическая регрессия обучается отличать ячейки,
#    связанные с известными проявлениями/аномалиями, от надежно удаленных ячеек;
# 4) итоговый прогноз = model.predict_proba(...), то есть ML-вероятность.

grid["target"] = np.nan
grid["evidence_score"] = 0.0
positive_cells = []
use_supervised = False
cv_roc_auc = None
cv_pr_auc = None

if USE_SUPERVISED and points is not None and len(points) > 0:
    point_union = unary_union(points.geometry)
    dist_to_evidence = np.array([geom.centroid.distance(point_union) for geom in grid.geometry], dtype=float)
    radius = CELL_SIZE * EVIDENCE_RADIUS_CELLS

    evidence_score = np.exp(-dist_to_evidence / radius)
    evidence_score = robust_normalize_01(evidence_score, 0.02, 0.995)
    grid["evidence_score"] = evidence_score

    try:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", predicate="within")
    except Exception:
        joined = gpd.sjoin(points[["geometry"]], grid[["cell_id", "geometry"]], how="left", op="within")

    positive_cells = joined["cell_id"].dropna().astype(int).unique().tolist()
    if positive_cells:
        grid.loc[grid["cell_id"].isin(positive_cells), "evidence_score"] = 1.0

    pos_thr = float(np.nanquantile(grid["evidence_score"], POSITIVE_EVIDENCE_Q))
    pos_mask = (grid["evidence_score"] >= pos_thr) | grid["cell_id"].isin(positive_cells)

    neg_thr = float(np.nanquantile(grid["evidence_score"], NEGATIVE_EVIDENCE_Q))
    neg_mask = (grid["evidence_score"] <= neg_thr) & (grid["geo_score_sm"] <= grid["geo_score_sm"].quantile(0.65))

    grid.loc[pos_mask, "target"] = 1
    grid.loc[neg_mask, "target"] = 0

    if grid["target"].eq(1).sum() < 10:
        pos_thr = float(np.nanquantile(grid["evidence_score"], 0.80))
        grid.loc[grid["evidence_score"] >= pos_thr, "target"] = 1

    use_supervised = (grid["target"].eq(1).sum() >= 3) and (grid["target"].eq(0).sum() >= 3)

if not use_supervised:
    grid["target"] = np.nan
    grid.loc[grid["geo_score_sm"] >= grid["geo_score_sm"].quantile(0.88), "target"] = 1
    grid.loc[grid["geo_score_sm"] <= grid["geo_score_sm"].quantile(0.25), "target"] = 0

# Нелинейные признаки.
base_features = [
    "prox_facies", "prox_paleo", "prox_struct", "prox_magm", "prox_tect1", "prox_tect2",
]

for col in base_features:
    grid[f"{col}_sqrt"] = np.sqrt(np.clip(grid[col], 0, 1))
    grid[f"{col}_sq"] = np.clip(grid[col], 0, 1) ** 2

# Взаимодействия факторов.
grid["i_tects"] = grid["prox_tect1"] * grid["prox_tect2"]
grid["i_tect_magm"] = grid["tect_combo"] * grid["prox_magm"]
grid["i_tect_struct"] = grid["tect_combo"] * grid["prox_struct"]
grid["i_paleo_struct"] = grid["prox_paleo"] * grid["prox_struct"]
grid["i_facies_paleo"] = grid["prox_facies"] * grid["prox_paleo"]
grid["i_complex"] = (
    grid["tect_combo"]
    * (0.5 * grid["prox_struct"] + 0.5 * grid["prox_magm"])
    * (0.5 * grid["prox_paleo"] + 0.5 * grid["prox_facies"])
)

grid["max_base_prox"] = grid[base_features].max(axis=1)
grid["mean_base_prox"] = grid[base_features].mean(axis=1)
grid["min_base_prox"] = grid[base_features].min(axis=1)
grid["count_high_prox"] = (grid[base_features] >= 0.65).sum(axis=1) / len(base_features)

feature_cols = (
    base_features
    + [f"{c}_sqrt" for c in base_features]
    + [f"{c}_sq" for c in base_features]
    + [
        "tect_combo", "tect_intersection", "tect_magm_intersection",
        "tect_struct_intersection", "paleo_struct_intersection", "coincidence_score",
        "tect_only_penalty", "i_tects", "i_tect_magm", "i_tect_struct",
        "i_paleo_struct", "i_facies_paleo", "i_complex",
        "max_base_prox", "mean_base_prox", "min_base_prox", "count_high_prox",
    ]
)

train_mask = grid["target"].notna()
X_train_full = grid.loc[train_mask, feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()
y_train_full = grid.loc[train_mask, "target"].astype(int).to_numpy()
X_all = grid[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()

sample_weight = 1.0 + 3.0 * grid.loc[train_mask, "evidence_score"].fillna(0).to_numpy()

logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        penalty="l2",
        C=LOGREG_C,
        class_weight="balanced",
        solver="lbfgs",
        max_iter=LOGREG_MAX_ITER,
        random_state=RANDOM_STATE,
    ),
)

try:
    n_pos = int((y_train_full == 1).sum())
    n_neg = int((y_train_full == 0).sum())
    n_splits = max(2, min(5, n_pos, n_neg))
    if n_splits >= 2:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
        cv_pred = cross_val_predict(logreg, X_train_full, y_train_full, cv=cv, method="predict_proba")[:, 1]
        cv_roc_auc = float(roc_auc_score(y_train_full, cv_pred))
        cv_pr_auc = float(average_precision_score(y_train_full, cv_pred))
except Exception:
    cv_roc_auc = None
    cv_pr_auc = None

logreg.fit(X_train_full, y_train_full, logisticregression__sample_weight=sample_weight)

grid["logreg_raw"] = np.clip(logreg.predict_proba(X_all)[:, 1], 0, 1)
grid["logreg_score"] = robust_normalize_01(grid["logreg_raw"], 0.02, 0.98)
grid["logreg_score_gamma"] = np.clip(grid["logreg_score"], 0, 1) ** PROSPECTIVITY_GAMMA
grid["logreg_score_sm"] = robust_normalize_01(
    smooth_on_regular_grid(grid, "logreg_score_gamma", grid_shape, passes=ML_SMOOTH_PASSES),
    0.02,
    0.98,
)

coef = logreg.named_steps["logisticregression"].coef_[0]
coef_table = pd.DataFrame({"feature": feature_cols, "coef": coef})
coef_table["abs_coef"] = coef_table["coef"].abs()
coef_table = coef_table.sort_values("abs_coef", ascending=False)
feature_importance = dict(zip(coef_table["feature"], coef_table["coef"].astype(float)))

In [ ]:
# %% Cell 8
# =========================
# FINAL SURFACE
# =========================

# Итоговая поверхность строится ML-моделью:
# prospectivity = Regularized Logistic Regression predict_proba после трансформаций признаков.
grid["prospectivity_raw"] = grid["logreg_score_sm"]

# Небольшой штраф у границ нужен только против краевых артефактов.
mask_boundary = mask_union.boundary
grid["dist_to_boundary"] = np.array([geom.distance(mask_boundary) for geom in grid.geometry])
edge_near = np.exp(-grid["dist_to_boundary"].to_numpy() / (CELL_SIZE * 2.2))
grid["edge_false_penalty"] = robust_normalize_01(
    edge_near * np.clip(grid["tect_only_penalty"], 0, 1),
    0.02,
    0.98,
)

grid["prospectivity_raw"] = grid["prospectivity_raw"] - 0.020 * grid["edge_false_penalty"]
grid["prospectivity_raw_sm"] = smooth_on_regular_grid(grid, "prospectivity_raw", grid_shape, passes=1)
grid["prospectivity"] = robust_normalize_01(grid["prospectivity_raw_sm"], 0.02, 0.98)

# Для совместимости с методичкой:
# в поле prognoz меньшее значение означает более перспективную ячейку.
grid["prognoz"] = 1.0 - grid["prospectivity"]

grid = make_display_classes(grid)
grid = mark_gold_zones(grid, grid_shape, mask_union)

In [ ]:
# %% Cell 9
# =========================
# SAVE
# =========================
if OUT_GPKG.exists():
    OUT_GPKG.unlink()

grid.to_file(OUT_GPKG, layer="forecast_grid", driver="GPKG")
if points is not None and len(points) > 0:
    points.to_file(OUT_GPKG, layer="evidence_points", driver="GPKG")

grid.drop(columns="geometry").to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
coef_table.to_csv(OUT_DIR / "logreg_coefficients.csv", index=False, encoding="utf-8-sig")

plot_prox(grid, mask, OUT_PROX)
plot_final(grid, mask, points, OUT_PNG)
plot_compare(grid, mask, points, OUT_COMPARE)

metrics = {
    "method": "Regularized Logistic Regression + nonlinear transformations",
    "base_dir": str(BASE_DIR),
    "grid_cells": int(len(grid)),
    "cell_size": CELL_SIZE,
    "use_supervised_requested": bool(USE_SUPERVISED),
    "use_supervised_applied": bool(use_supervised),
    "positive_cells": int(len(positive_cells)),
    "train_labeled_cells": int(train_mask.sum()),
    "train_positive_cells": int((y_train_full == 1).sum()),
    "train_negative_cells": int((y_train_full == 0).sum()),
    "cv_roc_auc": cv_roc_auc,
    "cv_pr_auc": cv_pr_auc,
    "logreg_C": LOGREG_C,
    "prospectivity_min": float(np.nanmin(grid["prospectivity"])),
    "prospectivity_p05": float(np.nanquantile(grid["prospectivity"], 0.05)),
    "prospectivity_p50": float(np.nanquantile(grid["prospectivity"], 0.50)),
    "prospectivity_p95": float(np.nanquantile(grid["prospectivity"], 0.95)),
    "prospectivity_max": float(np.nanmax(grid["prospectivity"])),
    "gold_zone_count": int(grid["gold_zone"].sum()),
    "gold_zone_share": float(grid["gold_zone"].mean()),
    "point_count": int(len(points)) if points is not None else 0,
    "logreg_coefficients": feature_importance,
}
OUT_JSON.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")

print("Готово.")
print(f"PNG: {OUT_PNG}")
print(f"COMPARE: {OUT_COMPARE}")
print(f"GPKG: {OUT_GPKG}")
print(f"CSV: {OUT_CSV}")
print(f"JSON: {OUT_JSON}")
print("CV ROC-AUC:", cv_roc_auc)
print("CV PR-AUC:", cv_pr_auc)
print("Топ коэффициентов:")
print(coef_table.head(12))
print(grid[["prospectivity", "prognoz", "logreg_score_sm", "gold_zone"]].describe())